In [1]:
%pwd


'c:\\Users\\saipa\\Medical-chatbot\\research'

In [2]:
import os
os.chdir("../")

In [3]:
%pwd

'c:\\Users\\saipa\\Medical-chatbot'

In [9]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Helper function to load all PDFs from a folder
def load_pdf_files(data_path):
    loader = DirectoryLoader(
        data_path,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    documents = loader.load()
    return documents

# Splitter for chunking text
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)


extracted_data = load_pdf_files("data")

# Split into chunks
chunks = text_splitter.split_documents(extracted_data)

print(f"Total documents: {len(extracted_data)}")
print(f"Total chunks: {len(chunks)}")
print(chunks[0].page_content[:300])  # preview first chunk


Total documents: 5
Total chunks: 30
A DATASET OF ROBOT-PATIENT AND DOCTOR-PATIENT MEDICAL DIALOGUES FOR
SPOKEN LANGUAGE PROCESSING TASKS
Heriberto Cuay´ahuitl
School of Engineering and Physical Sciences
University of Lincoln, UK
Grace Jang
Lincoln Medical School
Universities of Lincoln and Nothingham, UK
ABSTRACT
Large Language Models


In [10]:
extracted_data

[Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '', 'author': 'Heriberto Cuayahuitl; Grace Jang', 'doi': 'https://doi.org/10.48550/arXiv.2605.26747', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'A Dataset of Robot-Patient and Doctor-Patient Medical Dialogues for Spoken Language Processing Tasks', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2605.26747v1', 'source': 'data\\medical bot data.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}, page_content='A DATASET OF ROBOT-PATIENT AND DOCTOR-PATIENT MEDICAL DIALOGUES FOR\nSPOKEN LANGUAGE PROCESSING TASKS\nHeriberto Cuay´ahuitl\nSchool of Engineering and Physical Sciences\nUniversity of Lincoln, UK\nGrace Jang\nLincoln Medical School\nUniversities of Lincoln and Nothingham, UK\nABSTRACT\nLarge Language Models (LLMs) have brought huge

In [11]:
len(extracted_data)

5

In [27]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of Document objects, return a new list of Document objects
    containing only 'source' in metadata and the original page_content.
    """
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

# Function call
minimal_docs = filter_to_minimal_docs(extracted_data)


In [28]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [29]:
minimal_docs

[Document(metadata={'source': 'data\\medical bot data.pdf'}, page_content='A DATASET OF ROBOT-PATIENT AND DOCTOR-PATIENT MEDICAL DIALOGUES FOR\nSPOKEN LANGUAGE PROCESSING TASKS\nHeriberto Cuay´ahuitl\nSchool of Engineering and Physical Sciences\nUniversity of Lincoln, UK\nGrace Jang\nLincoln Medical School\nUniversities of Lincoln and Nothingham, UK\nABSTRACT\nLarge Language Models (LLMs) have brought huge improve-\nments to Artificial Intelligence (AI), which can be applied\nto general-purpose tasks. However, their application to tex-\ntual or spoken medical consultations is still an open research\nproblem. This paper proposes MeDial-Speech, a novel speech\ndataset for training and evaluating Med-AIs that can carry out\nconsultations with patients. It was collected in realistic en-\nvironments from robot-patient and doctor-patient dialogues,\ncontains 111+ hours of speech data (without data augmenta-\ntion), and covers four health conditions: Lewy body demen-\ntia, heart failure, shou

In [30]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Split the document into smaller chunks
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
    )
    text_chunks = text_splitter.split_documents(minimal_docs)
    return text_chunks


In [35]:
text_chunks = text_split(minimal_docs)
print(f"number of chunks: {len(text_chunks)}")


number of chunks: 52


In [ ]:
text_chunks

In [43]:
from langchain_community.embeddings import HuggingFaceEmbeddings

def download_embedding():
    """
    Download and return the HuggingFace embedding model.
    """
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embedding = HuggingFaceEmbeddings(
        model_name=model_name
    )
    return embedding


embeddings = download_embedding()


In [44]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [45]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [51]:
from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)





In [53]:
from pinecone import ServerlessSpec

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,  
        metric="cosine",  
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
index = pc.Index(index_name)


In [57]:
from langchain_pinecone import PineconeVectorStore

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,          
    embedding=embeddings,       
    index_name=index_name       
)


In [61]:
# load existing index
from langchain_pinecone import PineconeVectorStore

# Embed each chunk and upsert embedding into your Pinecone index
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)


In [62]:
# Convert docsearch into retriever
retriever = docsearch.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)


In [66]:
retrieved_doc = retriever.invoke("“I have chest pain, could this be angina?")
retrieved_doc

[Document(id='dd750a33-b79e-471f-9d34-32a4ca4dbabb', metadata={'source': 'data\\medical bot data.pdf'}, page_content='doctor-patient dialogues, covering four health conditions:\nLewy body dementia, heart failure, shoulder pain, and angina.\nThe former included in-person and remote consultations,\nwhereas the latter were carried out face-to-face. About 70%\nof spoken dialogues have been annotated with manual tran-\nscriptions of doctor and patient utterances, and also with\nspeaker role labels (doctor, robot, patient). It aims to be a\nuseful resource for AI practitioners to develop or evaluate'),
 Document(id='5eb9476e-6e7f-4841-8cdc-c15378785453', metadata={'source': 'data\\medical bot data.pdf'}, page_content='tual or spoken medical consultations is still an open research\nproblem. This paper proposes MeDial-Speech, a novel speech\ndataset for training and evaluating Med-AIs that can carry out\nconsultations with patients. It was collected in realistic en-\nvironments from robot-pati

In [67]:
# Load ChatOpenAI model
from langchain_openai import ChatOpenAI

ChatModel = ChatOpenAI(model="gpt-4o")


In [68]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate


In [69]:
system_prompt = (
    "You are 'Medical-Chatbot' — a compassionate, reliable, and knowledgeable AI assistant.\n"
    "Your role is to provide clear, accurate, and supportive medical guidance based on retrieved documents.\n"
    "Always follow these principles:\n\n"
    "1. Empathy First: Respond with care, respect, and supportive tone.\n"
    "2. Clarity: Use simple, easy-to-understand language (English + Hindi/Marathi mix if needed).\n"
    "3. Accuracy: Base answers strictly on retrieved medical documents and embeddings.\n"
    "4. Boundaries: Remind users you are not a doctor; for emergencies, advise immediate hospital visit.\n"
    "5. Structure:\n"
    "   - Summarize findings clearly.\n"
    "   - Highlight possible causes or remedies.\n"
    "   - Suggest next steps (consultation, lifestyle tips, etc.).\n"
    "6. Tone: Professional yet friendly — like a trusted medical companion.\n\n"
    "Your goal is to make healthcare information accessible, empathetic, and actionable.\n\n"
    "{context}"
)

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)


In [83]:
# Model initialization
from langchain_openai import ChatOpenAI
chatmodel = ChatOpenAI(model="gpt-4o")   

question_answer_chain = create_stuff_documents_chain(chatmodel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)




In [85]:

response = rag_chain.invoke({"input": "I get chest pain when I walk fast, is this angina?"})
print(response["answer"])


RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}